# Appendix A8: all models get misaligned

Reproduces Figure A8 ("All models we studied get misaligned when finetuned
on wrong advice") of `Unequal_influence.pdf` - a pure byproduct of
`cross_model_figure5_<dataset>.yaml`'s per-model baseline training. No new
training jobs beyond what Figure 5 already runs.

**Prerequisite**:

```bash
cd finetuning
em-influence run experiments/cross_model_figure5_auto.yaml --resume
em-influence run experiments/cross_model_figure5_career.yaml --resume
em-influence run experiments/cross_model_figure5_edu.yaml --resume
```

The paper's dashed "Pre-finetune rate" reference line needs one extra
one-off evaluation *per model* (11 total) of that model un-fine-tuned - see
`appendix_scores.ipynb`'s prerequisite cell for the pattern
(`--model-kind base`, no LoRA). Point `PRETRAIN_RATES` below at a
`{model_id: misaligned_pct}` mapping built from those; leave it empty to
plot without the reference lines.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_ROOTS = {
    "auto": Path("../../results/em_influence/filter_sweep_auto"),
    "career": Path("../../results/em_influence/filter_sweep_career"),
    "edu": Path("../../results/em_influence/filter_sweep_edu"),
}
OUTPUT_DIR = RESULTS_ROOTS["career"] / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MISALIGNED_THRESHOLD = 3
DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}

MODEL_LABELS = {
    "allenai/Olmo-3-7B-Instruct-SFT": "OLMo 3 7B",
    "Qwen/Qwen2.5-1.5B-Instruct": "Qwen 2.5 1B",
    "Qwen/Qwen2.5-3B-Instruct": "Qwen 2.5 3B",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen 2.5 7B",
    "Qwen/Qwen2.5-14B-Instruct": "Qwen 2.5 14B",
    "Qwen/Qwen3-4B": "Qwen 3 4B",
    "Qwen/Qwen3-8B": "Qwen 3 8B",
    "Qwen/Qwen3-14B": "Qwen 3 14B",
    "meta-llama/Llama-3.2-1B-Instruct": "Llama 3.2 1B",
    "meta-llama/Llama-3.2-3B-Instruct": "Llama 3.2 3B",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama 3.1 8B",
}
# Matches cross_model_figure5_<dataset>.yaml's `cross_model.models` order,
# which is itself the paper's Figure 5/A8/A9-A11 model ordering.
MODEL_ORDER = list(MODEL_LABELS)

PRETRAIN_RATES = {}  # {model_id: misaligned_pct}, see the prerequisite cell above


## Load every model's baseline

`cross_model_sweep`'s baseline train jobs carry a `model` parameter (the
model_id) that `filter_sweep`'s single-model baseline doesn't need - that's
how this pulls out one row per model instead of one row total.

In [ ]:
def load_all_model_baselines(results_root, dataset):
    manifest_path = results_root / "manifest.csv"
    df = pd.read_csv(manifest_path)
    baseline = df[(df["mode"] == "none") & df["model"].notna()]
    rows = []
    for model, group in baseline.groupby("model"):
        answers = pd.concat([pd.read_csv(csv) for csv in group["answers_csv"]], ignore_index=True)
        rate = 100 * (answers["aligned"] < MISALIGNED_THRESHOLD).mean()
        rows.append({"dataset": dataset, "model": model, "misaligned_pct": rate})
    return pd.DataFrame(rows)


## Figure A8

In [ ]:
def plot_all_models_misaligned(datasets=("auto", "career", "edu"), pretrain_rates=PRETRAIN_RATES):
    rates = pd.concat([load_all_model_baselines(RESULTS_ROOTS[d], d) for d in datasets], ignore_index=True)
    present = [model for model in MODEL_ORDER if model in rates["model"].unique()]

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(datasets))
    width = 0.8 / len(present)
    palette = plt.cm.tab20(np.linspace(0, 1, len(present)))
    for i, model in enumerate(present):
        values = [rates[(rates["dataset"] == d) & (rates["model"] == model)]["misaligned_pct"].mean() for d in datasets]
        offsets = x + (i - (len(present) - 1) / 2) * width
        ax.bar(offsets, values, width=width * 0.9, color=palette[i], label=MODEL_LABELS[model])
        if model in pretrain_rates:
            for xi in offsets:
                ax.hlines(pretrain_rates[model], xi - width * 0.45, xi + width * 0.45, color="black", linewidth=1)

    ax.set_xticks(x)
    ax.set_xticklabels([DATASET_LABELS.get(d, d) for d in datasets])
    ax.set_ylabel("Misaligned completions (%)")
    ax.set_xlabel("Dataset")
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8)
    fig.tight_layout()
    return fig


fig = plot_all_models_misaligned()
fig.savefig(OUTPUT_DIR / "figure_a8_all_models_misaligned.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure_a8_all_models_misaligned.pdf", bbox_inches="tight")
print(f"Saved to {OUTPUT_DIR / 'figure_a8_all_models_misaligned.png'}")
